In [2]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

# ============================
# 1. LOAD DATASET
# ============================
df = pd.read_csv(r"E:\Internship_project\my_ai_project\my_ai_project\crop\crop-training\fertilizer_recommendation.csv")

# ============================
# 2. HANDLE CATEGORICAL DATA
# ============================
categorical_cols = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage",
    "Season", "Irrigation_Type", "Previous_Crop",
    "Region", "Fertilizer_Used_Last_Season"
]

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# ============================
# 3. TARGET ENCODING
# ============================
target_encoder = LabelEncoder()
df["Recommended_Fertilizer"] = target_encoder.fit_transform(df["Recommended_Fertilizer"])

selected_features = [
    "Soil_Type",
    "Soil_pH",
    "Soil_Moisture",
    "Nitrogen_Level",
    "Phosphorus_Level",
    "Potassium_Level",
    "Temperature",
    "Humidity",
    "Crop_Type"
]

X = df[selected_features]
y = df["Recommended_Fertilizer"]

# ============================
# 5. TRAIN TEST SPLIT
# ============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ============================
# 6. SCALING
# ============================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================
# 7. ONE HOT ENCODING (TARGET)
# ============================
num_classes = len(np.unique(y))
y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)

# ============================
# 8. BUILD ANN MODEL
# ============================
model = Sequential()

model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(32, activation='relu'))

model.add(Dense(num_classes, activation='softmax'))

# ============================
# 9. COMPILE
# ============================
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ============================
# 10. TRAIN
# ============================
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_test, y_test)
)

# ============================
# 11. EVALUATE
# ============================
loss, acc = model.evaluate(X_test, y_test)
print(f"\n🔥 Accuracy: {acc * 100:.2f}%")

# ============================
# 12. SAVE MODEL + OBJECTS
# ============================
model.save("fertilizer_model.h5")

pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(encoders, open("encoders.pkl", "wb"))
pickle.dump(target_encoder, open("target_encoder.pkl", "wb"))

print("✅ Model + Encoders Saved Successfully!")

e:\Internship_project\.venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.6611 - loss: 0.9627 - val_accuracy: 0.7890 - val_loss: 0.6463
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7739 - loss: 0.6640 - val_accuracy: 0.7995 - val_loss: 0.5833
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7983 - loss: 0.5917 - val_accuracy: 0.8110 - val_loss: 0.5358
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8091 - loss: 0.5562 - val_accuracy: 0.8215 - val_loss: 0.5045
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8151 - loss: 0.5204 - val_accuracy: 0.8260 - val_loss: 0.4855
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8299 - loss: 0.4891 - val_accuracy: 0.8340 - val_loss: 0.4684
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8372 - loss: 0.4758 - val_accuracy: 0.8425 - val_loss: 0.4486
Epoch 8/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8453 - loss: 0.4569 - val_accuracy: 0.


🔥 Accuracy: 85.85%
✅ Model + Encoders Saved Successfully!
